[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://drive.google.com/file/d/1O19PKdGzkc8Uqa6xpZ0wH9yBa2sT2eqB/view?usp=sharing)

# Prompt Evaluation – Custom Metrics

This notebook demonstrates how to add custom metrics and criteria-based scoring to prompt evaluations. You can use function-based metrics or LLM-as-judge criteria to compare prompt quality.

**Objectives**
- Install Floeval and configure credentials
- Provide paths to prompts YAML and partial dataset JSON
- Define a custom metric for prompt evaluation
- Use `criteria()` for LLM-as-judge scoring
- Run evaluation and compare results across prompts

## 1. Installation

Install Floeval before running this notebook.

In [ ]:
%pip install floeval>=0.2.0b1

## 2. Configuration Constants

Set the following constants before running. Replace placeholder values with your API credentials and model identifiers. Required only for the optional criteria-based metric; function-based metrics do not require an API key.

**Provider flexibility:** You can use any OpenAI-compatible provider (OpenAI, Azure OpenAI, Anthropic, local models, etc.) — set the appropriate `base_url` and model names for your provider.

**Using FloTorch:** If you want to use FloTorch keys and gateway, obtain credentials from the [FloTorch Console](https://docs.flotorch.cloud/introduction/).

In [ ]:
import getpass
# LLM and API configuration (OpenAI)

OPENAI_BASE_URL = "https://api.openai.com/v1"
OPENAI_API_KEY = getpass.getpass("Enter your API key: ")
OPENAI_CHAT_MODEL = "gpt-4o-mini"
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"

## 3. Imports

Import Floeval evaluation classes, custom metric helpers, and provider configuration schema.

In [ ]:
from pathlib import Path

from floeval import Evaluation, DatasetLoader
from floeval.api.metrics.custom import custom_metric, criteria
from floeval.config.schemas.io.llm import OpenAIProviderConfig

## 4. Define a Custom Metric (Response Length)

Define a function-based metric with `@custom_metric`; this metric does not require an API key.

In [ ]:
@custom_metric(name="response_length", threshold=0.3)
def response_length(response: str) -> float:
    """Score 0–1 based on response length (capped at 100 chars)."""
    return min(len(response) / 100.0, 1.0)

## 5. Define a Criteria-Based Metric (Empathy)

Use `criteria()` to define an LLM-as-judge rubric. This metric requires `llm_config` and an API key.

In [ ]:
empathy = criteria(
    name="empathy",
    description="Rate empathy from 0 to 1. Reward acknowledgment and supportive tone.",
    threshold=0.6,
)

## 6. Load Files and Run Evaluation

**Prompts YAML** + **dataset JSON** (same format as other prompt notebooks):

```yaml
prompts:
  "1":
    template: "..."
```

```json
{
  "samples": [ { "user_input": "...", "prompt_ids": ["1"] } ]
}
```

**Example files**  
<a href="../datasets/prompt_evaluation/sample_prompts_custom.yaml" download="sample_prompts_custom.yaml">sample_prompts_custom.yaml</a><br>
<a href="../datasets/prompt_evaluation/sample_prompt_partial_custom_metrics.json" download="sample_prompt_partial_custom_metrics.json">sample_prompt_partial_custom_metrics.json</a>

Provide prompts and dataset paths, load the dataset, configure `llm_config`, and run evaluation.

In [ ]:
try:
    from google.colab import files
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

if _IN_COLAB:
    print("Upload prompts YAML file:")
    uploaded_prompts = files.upload()
    if not uploaded_prompts:
        raise RuntimeError("No prompts file uploaded.")
    prompts_path = Path(next(iter(uploaded_prompts.keys())))
    print("Upload dataset JSON file:")
    uploaded_ds = files.upload()
    if not uploaded_ds:
        raise RuntimeError("No dataset file uploaded.")
    dataset_path = Path(next(iter(uploaded_ds.keys())))
else:
    prompts_path = Path(input("Enter path to prompts YAML file: ").strip().strip('"')).expanduser()
    dataset_path = Path(input("Enter path to dataset JSON file: ").strip().strip('"')).expanduser()


### Resolve Prompts and Dataset Paths

Provide `prompts_path` and `dataset_path` via uploads in Colab or local path input in Jupyter.


In [ ]:
dataset = DatasetLoader.from_file(dataset_path, partial_dataset=True)
print("Prompts:", prompts_path)
print("Dataset:", dataset_path, f"— {len(dataset.samples)} samples")

### Load Prompt Dataset

Load the partial prompt dataset with `DatasetLoader.from_file(..., partial_dataset=True)`.


In [ ]:
llm_config = OpenAIProviderConfig(
    base_url=OPENAI_BASE_URL,
    api_key=OPENAI_API_KEY,
    chat_model=OPENAI_CHAT_MODEL,
    embedding_model=OPENAI_EMBEDDING_MODEL,
)


### Configure LLM Provider

Build `OpenAIProviderConfig` for metrics that call your provider.


In [ ]:
evaluation = Evaluation(
    dataset=dataset,
    llm_config=llm_config,
    metrics=["custom:response_length", empathy],
    default_provider="ragas",
    dataset_generator_model=OPENAI_CHAT_MODEL,
    prompts_file=str(prompts_path),
)


### Build Evaluation Object

Configure `Evaluation` with custom metric(s), criteria metric(s), and prompt variants.


In [ ]:
results = evaluation.run()
print("Aggregate scores:", results.aggregate_scores)


### Run Evaluation

Execute `evaluation.run()` to compute and print aggregate metric scores.


## 7. Inspect Results by Prompt

Each result includes `prompt_id`, allowing comparison of custom and criteria metric behavior across prompt variants.

### Inspect per-sample results

Iterates `results.sample_results` to print each question snippet and metric scores.


In [ ]:
for sr in results.sample_results:
    pid = sr.get("prompt_id", "unknown")
    metrics = sr.get("metrics", {})
    print(f"Prompt: {pid}")
    for k, v in metrics.items():
        print(f"  {k}: {v.get('score')}")

## Summary

This notebook demonstrated how to add custom and criteria-based metrics to prompt evaluations.

The key components included:

1. **Prompts File**: Paths pointed to a YAML file with prompt variants for support-style scenarios.
2. **Custom Metric**: A function-based `response_length` metric was defined using `@custom_metric`.
3. **Criteria-Based Metric**: The `criteria()` helper was used to define an `empathy` LLM-as-judge metric.
4. **Evaluation Execution**: The prompt evaluation was run with both metrics and aggregate scores were inspected.

This example showcases combining custom, criteria-based, and built-in metrics in prompt evaluation.